In [5]:
!pip install discord

In [6]:
!unzip ./converted_keras.zip

Archive:  ./converted_keras.zip
replace keras_model.h5? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
 extracting: keras_model.h5          
replace labels.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
 extracting: labels.txt              


In [7]:
# importamos la libreria PIL: Python Imaging Library
# es necesario para poder abrir la imagen a analizar
import PIL
from PIL import Image, ImageOps
import numpy as np
# Instalamos keras, libreria necesaria para que funcionen
# las redes neuronales
!pip install -q tf-keras==2.19.0 h5py==3.11.0
# Importando tf-keras – una versión de Keras compatible con modelos .h5
import tf_keras as keras
# Importando la función load_model de tf_keras, que nos
# permitira cargar el modelo de google teachable machine
from tf_keras.models import load_model

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement tensorflow<2.20,>=2.19 (from tf-keras) (from versions: 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow<2.20,>=2.19


In [8]:
import discord
from discord.ext import commands
import nest_asyncio

nest_asyncio.apply()

In [9]:
model_path = "./keras_model.h5"
labels_path = "./labels.txt"

def get_class(image_path):
  np.set_printoptions(suppress=True)
  model = load_model(model_path, compile=False)
  class_names = open(labels_path, "r").readlines()
  data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
  image = Image.open(image_path).convert("RGB")
  size = (224, 224)
  image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)
  image_array = np.asarray(image)
  normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1
  data[0] = normalized_image_array
  prediction = model.predict(data)
  print(prediction)
  index = np.argmax(prediction)
  class_name = class_names[index]
  confidence_score = prediction[0][index]
  text1 = f"Class: {class_name[2:]} Confidence Score: {confidence_score}"
  return text1

In [12]:
#Definimos los permisos del bot
intents = discord.Intents.default()
intents.message_content = True
#creamos el bot
bot = commands.Bot(command_prefix='$', intents=intents)

@bot.event
async def on_ready():
    print(f'We have logged in as {bot.user}')
#este comando saluda al usuario
#para activarlo se escribe $hello
@bot.command()
async def hello(ctx):
    await ctx.send(f'Hi! I am a bot {bot.user}!')

#escribe tu codigo aqui
@bot.command()
async def check(ctx):
  if ctx.message.attachments:
    for archivo in ctx.message.attachments:
      file_name = archivo.filename
      file_url = archivo.url
      await archivo.save(f"./{file_name}")
      await ctx.send("Imagen guardada con exito.")
      respuesta = get_class(f"./{file_name}")
      await ctx.send(respuesta)
  else:
    await ctx.send("No has enviado ninguna imagen")


bot.run("TOKEN")



2026-08-27 23:44:21 INFO     discord.client logging in using static token
2026-08-27 23:44:21 INFO     discord.client logging in using static token
2026-08-27 23:44:21 INFO     discord.client logging in using static token
INFO:discord.client:logging in using static token
2026-08-27 23:44:22 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: 18db81fac514cc00cb95b17c150427d5).
2026-08-27 23:44:22 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: 18db81fac514cc00cb95b17c150427d5).
2026-08-27 23:44:22 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: 18db81fac514cc00cb95b17c150427d5).
INFO:discord.gateway:Shard ID None has connected to Gateway (Session ID: 18db81fac514cc00cb95b17c150427d5).


We have logged in as Botsito#1592
1/1 [==============================] - 2s 2s/step
[[0.06123003 0.00586868 0.05889393 0.01671703 0.23140574 0.03604319
  0.02013614 0.03962282 0.00335055 0.07778822 0.02987157 0.01017082
  0.1529531  0.03375834 0.08928497 0.04922433 0.00481161 0.03872727
  0.01678815 0.00931452 0.00354946 0.0038792  0.00147491 0.00513545]]
